In [0]:
VECTOR_DB_PATH = "/local_disk0/tmp/chroma_db_test/"


In [0]:
%pip install sentence-transformers chromadb

In [0]:
from sentence_transformers import SentenceTransformer
import chromadb
import os


In [0]:
GOLD_PATH = "/Volumes/workspace/legal_data/gold/legal_chunks/"

gold_df = spark.read.format("delta").load(GOLD_PATH)

gold_df = gold_df.select(
    "chunk_id",
    "chunk_text",
    "act_name",
    "section_number",
    "category",
    "file_name"
)

gold_df.display()

In [0]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [0]:
os.makedirs(VECTOR_DB_PATH, exist_ok=True)

client = chromadb.PersistentClient(path=VECTOR_DB_PATH)

collection = client.get_or_create_collection(
    name="legal_knowledge",
    metadata={"hnsw:space": "cosine"}
)

print(f"Collection ready. Existing vectors: {collection.count()}")


In [0]:
def safe_str(value):
   if value is None:
       return ""
   return str(value)

In [0]:
def upsert_batch(batch_rows):
    texts = []
    ids = []
    metadatas = []

    for r in batch_rows:
        text = safe_str(r.chunk_text).strip()
        if not text:
            continue

        texts.append(text)
        ids.append(safe_str(r.chunk_id))
        metadatas.append({
            "act_name": safe_str(r.act_name),
            "section": safe_str(r.section_number),
            "category": safe_str(r.category),
            "source": safe_str(r.file_name)
        })

    if not ids:
        return 0

    embeddings = model.encode(texts).tolist()

    collection.upsert(
        ids=ids,
        documents=texts,
        embeddings=embeddings,
        metadatas=metadatas
    )

    return len(ids)

batch_size = 100
batch = []
total = 0

for row in gold_df.toLocalIterator():
    batch.append(row)

    if len(batch) == batch_size:
        total += upsert_batch(batch)
        print(f"Processed {total} vectors")
        batch = []

if batch:
    total += upsert_batch(batch)

print(f"Done. Total vectors in collection: {collection.count()}")


In [0]:
query = "What is the penalty under Motor Vehicles Act for not wearing helmet under section 129?"

if collection.count() == 0:
    print("No vectors found. Run embedding ingestion first.")
else:
    query_embedding = model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=10
    )

    docs = results["documents"][0]
    filtered_docs = [doc for doc in docs if "helmet" in doc.lower()]

    print(filtered_docs[:3] if filtered_docs else docs[:3])
